In [ ]:
#| default_exp logview

# View logs

## Imports

In [ ]:
#| export
import sys, re, json, builtins
from pathlib import Path
from datetime import datetime
from fastcore.script import call_parse
from fastcore.xtras import asdict
from fastlite import database
from shell_sage.config import ShellSageConfig, get_cfg
from shell_sage.core import log_path, Log, mk_db

In [ ]:
#| hide
from tempfile import TemporaryDirectory
from IPython.display import Markdown
from fastcore.test import *

## Helpers

In [ ]:
rr = [
    {
        'id': 1,
        'timestamp': '2026-04-27T10:00:00',
        'query': '<system_info>abc</system_info>\n<query>\nHow do I list files?\n</query>',
        'response': 'Use `ls -la`.',
        'model': 'claude',
        'mode': 'default',
    },
    {
        'id': 2,
        'timestamp': '2026-04-27T10:01:00',
        'query': 'Plain legacy query without XML tags',
        'response': 'Legacy response',
        'model': 'claude',
        'mode': 'default',
    },
    {
        "id":3,
        "timestamp":"2026-04-26T12:06:45.744887",
        "query":"<system_info>\n<system>superduper 3</system>\n<shell>/bin/zsh</shell>\n<aliases>\naliases\n</aliases>\n</system_info><terminal_history>\nline 1\nline 2\"\n⠋ Connecting...\n\n\n\n</terminal_history>\n<query>\n???\n</query>",
        "response":"Yes! I am.",
        "model":"claude-sonnet-4-5-20250929",
        "mode":"default"
    }
]

In [ ]:
#| export
_log_field_names = tuple(Log.__annotations__.keys())
_front_matter = ('id', 'timestamp', 'model', 'mode')

In [ ]:
_log_field_names

('id', 'timestamp', 'query', 'response', 'model', 'mode')

In [ ]:
db = mk_db()

In [ ]:
# db.close()

In [ ]:
#| export
def _q_1line(query, lim=100):
    m = re.search(r'<query>\s*(.*?)\s*</query>', query or '', re.DOTALL | re.IGNORECASE)
    s = re.sub(r'\s+', ' ', (m.group(1) if m else (query or '')).strip(), flags=re.S)
    return s[: lim] + '...' if len(s) > lim else s

In [ ]:
test_eq(_q_1line(rr[1]['query']), 'Plain legacy query without XML tags')
test_eq(_q_1line('<query>\n' + ('x'*120) + '\n</query>', lim=10), 'xxxxxxxxxx...')

In [ ]:
lg = Log(id=1, timestamp=datetime.now().isoformat(), query='Hi!', model='llama3.2', 
        response='I am ShellSage, a command-line teaching assistant!', mode='default')
asdict(lg)

{'id': 1,
 'timestamp': '2026-04-27T18:41:34.167418',
 'query': 'Hi!',
 'response': 'I am ShellSage, a command-line teaching assistant!',
 'model': 'llama3.2',
 'mode': 'default'}

In [ ]:
json.dumps(asdict(lg), default=str)

'{"id": 1, "timestamp": "2026-04-27T18:41:34.167418", "query": "Hi!", "response": "I am ShellSage, a command-line teaching assistant!", "model": "llama3.2", "mode": "default"}'

In [ ]:
#| export
def log2json(r, **kw): return json.dumps({k: r.get(k) for k in _log_field_names}, default=str)

In [ ]:
j = log2json(rr[0])
test_eq('\n' in j, False)
test_eq('\\n' in j, True)
j

'{"id": 1, "timestamp": "2026-04-27T10:00:00", "query": "<system_info>abc</system_info>\\n<query>\\nHow do I list files?\\n</query>", "response": "Use `ls -la`.", "model": "claude", "mode": "default"}'

<details class='context-details'><summary>context</summary>\n\n{ctx}\n\n</details>

In [ ]:
#| export
def log2fm(r, **kw):
    ks = [k for k in _front_matter if k in r]
    if not ks: return ''
    return f'''---\n{'\n'.join(f"{k}: {r.get(k)}" for k in ks)}\n---\n'''

In [ ]:
fm = log2fm(rr[0])
test_is('id: 1' in fm, True)
test_is('timestamp: 2026-04-27T10:00:00' in fm, True)

In [ ]:
#| export
def _parse_query(qry):
    mtch = re.search(r'\A(.*)\n<query>\s*(.*?)\s*</query>\Z', qry, re.DOTALL | re.IGNORECASE)
    ctx, qry = mtch.groups() if mtch else ('','')
    return ctx, qry

def _md_parts(r, frontmatter):
    fm = log2fm(r)+'\n\n' if frontmatter else ''
    ctx, qry = _parse_query(r.get('query', ''))
    res = r.get('response')
    return fm, ctx, qry, res

def log2md(r, frontmatter=False, context=False, **kw):
    fm, ctx, qry, res = _md_parts(r, frontmatter)
    if context and ctx:
        ctx = f'''```xml\n{ctx}\n```\n\n'''
        ctx = f'''<details class='context-details'><summary>context</summary>\n\n{ctx}\n\n</details>\n\n'''
    else: ctx = ''
    turn = f'''## AI Prompt\n{qry}\n## AI Response\n{res}'''
    return f"{fm}{ctx}{turn}"

In [ ]:
md_plain = log2md(rr[1], frontmatter=False, context=False)
test_is('## AI Response' in md_plain, True)
test_is("<details class='context-details'>" not in md_plain, True)

md_full = log2md(rr[0], frontmatter=True, context=True)
test_is(md_full.startswith('---'), True)
test_is("<details class='context-details'>" in md_full, True)

In [ ]:
print(log2fm(rr[2]))

---
id: 3
timestamp: 2026-04-26T12:06:45.744887
model: claude-sonnet-4-5-20250929
mode: default
---



In [ ]:
Markdown(log2md(rr[2], frontmatter=True, context=True))

---
id: 3
timestamp: 2026-04-26T12:06:45.744887
model: claude-sonnet-4-5-20250929
mode: default
---


<details class='context-details'><summary>context</summary>

```xml
<system_info>
<system>superduper 3</system>
<shell>/bin/zsh</shell>
<aliases>
aliases
</aliases>
</system_info><terminal_history>
line 1
line 2"
⠋ Connecting...



</terminal_history>
```



</details>

## AI Prompt
???
## AI Response
Yes! I am.

## Log View Command

In [ ]:
class FakeDB:
    def __init__(self, rows): self._rows = rows
    def q(self, sql, params=None):
        s = sql.lower()
        if 'count(*) as c' in s: return [{'c': len(self._rows)}]
        if 'order by timestamp asc' in s: return list(self._rows)
        if 'order by timestamp desc' in s:
            n = params[0]
            return list(reversed(self._rows))[:n]
        return list(self._rows)

fdb = FakeDB(rr)

In [ ]:
#| export
_pr = builtins.print
def _die(msg, code=2): _pr(msg, file=sys.stderr); raise SystemExit(code)

def _active_db_path(): return log_path / 'logs.db'

def _ls():
    log_path.mkdir(parents=True, exist_ok=True)
    for p in sorted(log_path.glob('*.db'), key=lambda x: x.name):
        st = p.stat()
        dt = datetime.fromtimestamp(st.st_mtime).isoformat()
        act = 'active' if p.name == 'logs.db' else '-'
        _pr(f"{p.name}	{act}	{st.st_size}	{dt}")

def _db(log_db):
    pth = (Path(log_db) if log_db else _active_db_path()).expanduser()
    dcfg = asdict(ShellSageConfig())
    log_en = bool(get_cfg().get('log', dcfg['log']))
    if not pth.exists():
        _pr('ssage_log: no log database at', pth, file=sys.stderr)
        if not log_en:
            _pr(
                '  Set `log = True` in ~/.config/shell_sage/shell_sage.conf and run `ssage` to create logs.',
                file=sys.stderr,
            )
        else: _pr('  Run `ssage` to create a log if this path is new.', file=sys.stderr)
        raise SystemExit(1)
    db = database(pth)
    db.logs = db.create(Log)
    c = (db.q('select count(*) as c from log')[0] or {'c': 0})['c']
    if c == 0:
        _pr(
            'ssage_log: log database is empty; run `ssage` to record entries (with `log = True` in your config).',
            file=sys.stderr,
        )
        if not log_en:
            _pr('  `log` is false in your config; enable it to save new sessions to the database.', file=sys.stderr)
        return
    return db

def _info(out, db):
    w = (open(out, 'w', encoding='utf-8') if out else None)
    f = w or sys.stdout
    try:
        rows = db.q('select * from log order by timestamp asc, id asc') or []
        for i, r in enumerate(rows, 1):
            _pr(f"{i}	{r.get('timestamp', '')}	{_q_1line(r.get('query', ''))}", file=f)
    finally:
        if w: w.close()

def _rows(all, last, db):
    if all: return list(db.q('select * from log order by timestamp asc, id asc') or [])
    if last < 1: _die('ssage_log: --last must be at least 1', 2)
    tail = db.q('select * from log order by timestamp desc, id desc limit ?', (last,)) or []
    tail.reverse()
    return tail

In [ ]:
test_eq([r['id'] for r in _rows(all=False, last=2, db=fdb)], [2,3])
test_eq(len(_rows(all=False, last=1, db=fdb)), 1)

In [ ]:
#| export
def _to_nb(rows, **kw):
    return {
        'cells': [{'cell_type': 'markdown', 'metadata': {}, 'source': log2md(r, **kw)} for r in rows],
        'metadata': {}, 'nbformat': 4, 'nbformat_minor': 5,
    }

def _show(all, last, format, out, db, frontmatter=False, context=False):
    rows = _rows(all, last, db)
    w = open(out, 'w', encoding='utf-8') if out else None
    f = w or sys.stdout
    fmt = log2json if format == 'json' else log2md
    try:
        if format in ('json', 'md'):
            for r in rows: f.write(fmt(r, frontmatter=frontmatter, context=context) + '\n')
        else:
            json.dump(_to_nb(rows, frontmatter=frontmatter, context=context), f, ensure_ascii=False)
            f.write('\n')
    finally:
        if w: w.close()

In [ ]:
nb = _to_nb(rr)
test_eq(len(nb['cells']), 3)
test_eq(nb['cells'][0]['cell_type'], 'markdown')
test_is('## AI Prompt' in nb['cells'][0]['source'], True)

In [ ]:
with TemporaryDirectory() as td:
    td = Path(td)

    out_json = td/'out.json'
    _show(all=True, last=1, format='json', out=str(out_json), db=fdb)
    test_eq(len(out_json.read_text().splitlines()), 3)

    out_md = td/'out.md'
    _show(all=False, last=1, format='md', out=str(out_md), db=fdb, frontmatter=True, context=True)
    test_is('## AI Prompt' in out_md.read_text(), True)

    out_nb = td/'out.ipynb'
    _show(all=True, last=1, format='nb', out=str(out_nb), db=fdb)
    nbj = json.loads(out_nb.read_text())
    test_eq(len(nbj['cells']), 3)

In [ ]:
#| export
def _prune_db():
    log_path.mkdir(parents=True, exist_ok=True)
    src = _active_db_path()
    if not src.exists(): _die(f'ssage_log: active database not found at {src}', 1)
    ts = datetime.now().strftime('%Y-%m-%d-%H%M%S')
    dst = log_path / f'logs-{ts}.db'
    i = 1
    while dst.exists():
        dst = log_path / f'logs-{ts}-{i}.db'
        i += 1
    src.rename(dst)
    mk_db()
    _pr(f'pruned {src.name} -> {dst.name}')

In [ ]:
_old_log_path, _old_mk_db = log_path, mk_db
try:
    with TemporaryDirectory() as td:
        td = Path(td)
        log_path = td
        (td/'logs.db').write_bytes(b'synthetic')

        def _mk_db_stub(): (td/'logs.db').write_bytes(b'')

        mk_db = _mk_db_stub
        _prune_db()

        test_is((td/'logs.db').exists(), True)
        test_eq(len([p for p in td.glob('logs-*.db')]), 1)
finally:
    log_path, mk_db = _old_log_path, _old_mk_db

pruned logs.db -> logs-2026-04-27-184134.db


In [ ]:
#| export
@call_parse
def ssage_log(
    log_db: str|None = None,  # Path to sqlite log database; default: active file at ~/.shell_sage/logs/logs.db
    ls: bool = False,  # List .db files in the log directory and exit
    info: bool = False,  # Print order, timestamp, and one-line user query for every row; omits other modes
    prune: bool = False,  # Rotate active logs.db to logs-YYYY-MM-DD-HHMMSS.db and recreate empty logs.db
    all: bool = False,  # Include all rows in time order, oldest first (overrides --last)
    last: int = 1,  # How many of the *most recent* rows to print, oldest of that set first, unless you pass --all
    format: str = 'json',  # output: json, md, or nb (for default export only; not --info)
    frontmatter: bool = False,  # For --format md, include frontmatter with id/timestamp/model/mode
    context: bool = False,  # For --format md, include captured context in a collapsible details block
    out: str|None = None,  # Write to this file instead of stdout
):
    "Print logged ShellSage Q&A from the sqlite log database."
    if format not in ('json', 'md', 'nb'): _die(f'ssage_log: --format must be json, md, or nb, not {format!r}', 2)
    if info and (all or any(a == '--last' for a in sys.argv[1:])):
        _die('ssage_log: do not use --info together with --all or --last', 2)
    if ls: return _ls()
    if prune: return _prune_db()
    if not (db := _db(log_db)): return
    if info: return _info(out, db)
    _show(all, last, format, out, db, frontmatter=frontmatter, context=context)

## export -

In [ ]:
#| hide
#| eval: false
from nbdev.doclinks import nbdev_export
nbdev_export()